In [1]:
# Install libraries
!pip install pandas
%pip install nbconvert


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from datetime import datetime
from decimal import Decimal, ROUND_HALF_UP

In [3]:
# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Get the file path of data
lesson_logs_path = os.path.join(notebook_dir, "data", "raw", "lesson_logs_messy.xlsx")
print(lesson_logs_path)

/Users/huien/Documents/opus-tuition/src/data_cleaning/data/raw/lesson_logs_messy.xlsx


#### Read the dataset

In [4]:
# Read the dataset
df = pd.read_excel(lesson_logs_path)

# Get first five row of the dataset
df.head()

,LOG-001,TAS-001,05/01/25,1.5,Present,Covered algebra ch.3,82.5
0,LOG-002,TAS-002,12/03/25,1.0,Present,Reading comprehension,45
1,LOG-003,TAS-003,14/02/25,2.0,Present,Timed practice paper,120
2,LOG-004,TAS-004,20/03/25,1.5,Absent,Student unwell — cancelled,TBC
3,LOG-005,TAS-001,08/01/25,1.5,Present,Quadratic equations,82.5
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Analyse the dataset

In [5]:
# Get the number of columns and rows of the dataset
df.shape

(17, 7)

In [6]:
# Get the column names
column_names = list(df.columns)
print(column_names)

['LOG-001', 'TAS-001', '05/01/25', 1.5, 'Present', 'Covered algebra ch.3', 82.5]


In [7]:
# No header, craft one header
df.columns = ["Log ID", "Assignment ID", "Session Date", "Duration (Hours)", "Attendance Status", "Session Notes", "Fees Charged"]

In [8]:
# View summary of dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Log ID             15 non-null     str    
 1   Assignment ID      15 non-null     str    
 2   Session Date       15 non-null     str    
 3   Duration (Hours)   14 non-null     float64
 4   Attendance Status  15 non-null     str    
 5   Session Notes      15 non-null     str    
 6   Fees Charged       15 non-null     object 
dtypes: float64(1), object(1), str(5)
memory usage: 1.1+ KB


In [9]:
# Identify duplicated rows
duplicate_rows = df[df.duplicated(keep=False)]
print(duplicate_rows)

  Log ID Assignment ID Session Date  Duration (Hours) Attendance Status  \
4    NaN           NaN          NaN               NaN               NaN   
9    NaN           NaN          NaN               NaN               NaN   

  Session Notes Fees Charged  
4           NaN          NaN  
9           NaN          NaN  


In [10]:
# Get distinct value for attendance status
attendance_status = list(df["Attendance Status"].unique())
print(attendance_status)

['Present', 'Absent', nan, 'Late', 'Absent-MC']


#### Clean dataset

In [11]:
# Remove duplicates, leaving only first occurence
df_cleaned = df.drop_duplicates(keep="first")

In [12]:
# Drop blank rows with all values are nulls
df_cleaned = df_cleaned.dropna(how="all")

In [13]:
# Fill null attendance status values with "Absent"
df_cleaned["Attendance Status"] = df_cleaned["Attendance Status"].fillna("Absent")

In [14]:
def parse_to_date(df: pd.DataFrame, col_name: str) -> pd.DataFrame:
    expected_formats = [
        "%Y-%m-%d",     # 2025-03-12
        "%m/%d/%Y",     # 01/05/2025
        "%d-%m-%Y",     # 25-04-2025
        "%d %b %Y",     # 12 Feb 2025
        "%B %d, %Y",    # July 7, 2025
        "%d-%b-%Y",     # 15-Oct-2025
        "%d/%m/%y"      # 12/03/25
    ]

    parsed_values = []
    error_log = []

    # Loop only through the target column
    for idx, raw_val in df[col_name].items():
        if pd.isna(raw_val) or str(raw_val).strip() in ["", "None", "NaN"]:
            parsed_values.append(pd.NaT)
            continue

        clean_val = str(raw_val).strip()
        success = False

        for fmt in expected_formats:
            try:
                parsed_values.append(datetime.strptime(clean_val, fmt))
                success = True
                break
            except ValueError:
                continue

        if not success:
            # Capture specific row index and offending value
            error_log.append(f"Row {idx}: '{raw_val}' cannot be parsed.")
            parsed_values.append(pd.NaT)

    # If any row failed, stop everything and throw a specific error
    if error_log:
        raise ValueError(
            f"Data Integrity Error in column [{col_name}].\n"
            f"Details:\n" + "\n".join(error_log)
        )

    # If all passed, overwrite only that single column with the datetime type
    df[col_name] = parsed_values
    return df

df_cleaned = parse_to_date(df_cleaned, "Session Date")

In [15]:
# Replace null values to 1.0 by default
df_cleaned["Duration (Hours)"] = df_cleaned["Duration (Hours)"].fillna(1)

In [16]:
# Extract numbers, periods, and commas. Remove commas.
df_cleaned["Duration (Hours)"] = (
    df_cleaned["Duration (Hours)"]
    .astype(str)
    .str.extract(r"([\d.,]+)")[0]
    .str.replace(",", "", regex=False)
)

# Convert to Decimal and strictly round to 1 decimal places
def to_rounded_decimal(val):
    if pd.isna(val) or val == "nan":
        return None
    # Quantum ensures exactly one decimal places (.1)
    return Decimal(val).quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)

df_cleaned["Duration (Hours)"] = df_cleaned["Duration (Hours)"].apply(to_rounded_decimal)

In [17]:
# Replaces rows containing only letters with 0
df_cleaned["Fees Charged"] = df_cleaned["Fees Charged"].replace(r'^[A-Za-z]+$', 0, regex=True)

In [18]:
# Extract numbers, periods, and commas. Remove commas.
df_cleaned["Fees Charged"] = (
    df_cleaned["Fees Charged"]
    .astype(str)
    .str.extract(r"([\d.,]+)")[0]
    .str.replace(",", "", regex=False)
)

# Convert to Decimal and strictly round to 2 decimal places
def to_rounded_decimal(val):
    if pd.isna(val) or val == "nan":
        return None
    # Quantum ensures exactly two decimal places (.01)
    return Decimal(val).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

df_cleaned["Fees Charged"] = df_cleaned["Fees Charged"].apply(to_rounded_decimal)

In [19]:
# Convert selected columns to the modern string dtype
df_cleaned = df_cleaned.astype({
    "Log ID": "string",
    "Assignment ID": "string",
    "Attendance Status": "string",
    "Session Notes": "string"
})

#### Validate dataset

In [20]:
# View summary of dataframe
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 15 entries, 0 to 16
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Log ID             15 non-null     string        
 1   Assignment ID      15 non-null     string        
 2   Session Date       15 non-null     datetime64[us]
 3   Duration (Hours)   15 non-null     object        
 4   Attendance Status  15 non-null     string        
 5   Session Notes      15 non-null     string        
 6   Fees Charged       15 non-null     object        
dtypes: datetime64[us](1), object(2), string(4)
memory usage: 960.0+ bytes


In [21]:
# Print the final cleaned results
df_cleaned

,Log ID,Assignment ID,Session Date,Duration (Hours),Attendance Status,Session Notes,Fees Charged
0,LOG-002,TAS-002,2025-03-12,1.0,Present,Reading comprehension,45.00
1,LOG-003,TAS-003,2025-02-14,2.0,Present,Timed practice paper,120.00
2,LOG-004,TAS-004,2025-03-20,1.5,Absent,Student unwell — cancelled,0.00
3,LOG-005,TAS-001,2025-01-08,1.5,Present,Quadratic equations,82.50
5,LOG-006,TAS-006,2025-04-26,1.0,Present,Organic chemistry intro,70.00
6,LOG-007,TAS-007,2025-06-03,1.5,Present,Newton's laws,75.00
7,LOG-008,TAS-008,2025-07-09,1.0,Present,Linear equations,45.00
8,LOG-009,TAS-003,2025-02-21,2.0,Present,Model essay writing,120.00
10,LOG-010,TAS-010,2025-08-22,1.5,Present,Cell biology revision,82.50
11,LOG-011,TAS-001,2025-01-15,1.5,Late,Student arrived 20 mins late,82.50
